# FedHealth — Notebook 02: Preprocessing & Normalization
Produces the artifacts every tier reuses: the **fitted preprocessor** (median imputer + StandardScaler) and the stratified splits. Fitting happens on the training split only, so nothing leaks into val/test — and the FL clients + inference service load this exact object, guaranteeing identical feature vectors across the federation.

In [ ]:
# --- Setup ------------------------------------------------------------------
# Make the shared fedhealth_ml package importable in Colab. Pick ONE option:
#
#   (A) Clone your project repo (recommended):
# !git clone https://github.com/<you>/fedhealth.git
# import sys; sys.path.append('/content/fedhealth/ml')
#
#   (B) Upload the `ml/` folder via the Files panel, then:
# import sys; sys.path.append('/content/ml')
#
#   (C) Mount Google Drive where you keep the repo:
# from google.colab import drive; drive.mount('/content/drive')
# import sys; sys.path.append('/content/drive/MyDrive/fedhealth/ml')
import sys
sys.path.append('/content/fedhealth/ml')  # <-- adjust to match your choice above

# Install anything Colab is missing (Colab already has numpy/pandas/sklearn/tf).
!pip -q install xgboost>=2.0 scipy joblib 2>/dev/null

# Where the three CSVs live. Upload them or point DATA_DIR at your Drive folder.
DATA_DIR = '/content'          # <-- put the CSVs here (or change this path)
ANEMIA = f'{DATA_DIR}/anemia_diagnosed_cbc_data_v4.csv'
THAL   = f'{DATA_DIR}/thalassemia_alphanorm.csv'
LEUK   = f'{DATA_DIR}/biased_leukemia_dataset.csv'

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from fedhealth_ml import schema, preprocess as pp
print('fedhealth_ml loaded. Model A features:', len(schema.MODEL_A_FEATURES))


## 1. Load + clean (impossible values -> NaN)

In [ ]:
X, y = pp.load_anemia(ANEMIA)
print('X', X.shape, '| NaNs introduced by range-cleaning:', int(X.isna().sum().sum()))
print('classes present:', sorted(y.unique()))

## 2. Stratified train / val / test split
Stratification preserves the (very imbalanced) class ratios in each split.

In [ ]:
sp = pp.stratified_split(X, y, test_size=0.2, val_size=0.1, seed=42)
for k in ['X_train','X_val','X_test']:
    print(f'{k:8}', sp[k].shape)

## 3. Fit the preprocessor on TRAIN only, then save it

In [ ]:
import os
ART = '/content/artifacts/modelA'; os.makedirs(ART, exist_ok=True)
prep = pp.CBCPreprocessor(schema.MODEL_A_FEATURES).fit(sp['X_train'])
prep.save(f'{ART}/preprocessor.joblib')
Xtr = prep.transform(sp['X_train']); Xval = prep.transform(sp['X_val']); Xte = prep.transform(sp['X_test'])
print('scaled train', Xtr.shape, '| any NaN:', bool(np.isnan(Xtr).any()))
print('saved preprocessor ->', f'{ART}/preprocessor.joblib')

## 4. Encode labels + inspect class imbalance
The class weights below are what the ANN uses to avoid biasing toward frequent classes (you asked for balanced performance + healthy detection).

In [ ]:
from fedhealth_ml import models
ytr = pp.encode_labels(sp['y_train'], schema.MODEL_A_CLASSES)
yval = pp.encode_labels(sp['y_val'], schema.MODEL_A_CLASSES)
yte = pp.encode_labels(sp['y_test'], schema.MODEL_A_CLASSES)
cw = models.class_weights(ytr, len(schema.MODEL_A_CLASSES))
for i, c in enumerate(schema.MODEL_A_CLASSES):
    print(f'{c:34} n_train={int((ytr==i).sum()):4d}  weight={cw[i]:.2f}')

## 5. Save arrays for the training notebook

In [ ]:
np.savez(f'{ART}/arrays.npz', Xtr=Xtr, Xval=Xval, Xte=Xte, ytr=ytr, yval=yval, yte=yte)
print('saved ->', f'{ART}/arrays.npz')

### Thalassemia preprocessing
Deferred until the Model B dataset decision (Notebook 04). The same `pp.load_thal` + `CBCPreprocessor(schema.MODEL_B_FEATURES)` pipeline applies to whichever dataset is chosen.